In [2]:
from langgraph.constants import START
from langgraph.graph import StateGraph

from typing import TypedDict
# 此脚本用于展示私有状态的设置。私有状态只用于内部消息传递，不参与输入和输出

# 主状态
class MyState(TypedDict):
    query:str
    final_answer:str

#私有状态,用于中间过程 的数据传递，不参与最终的输出
class PrivateState(TypedDict):
    rag_result:str
    web_result:str

class InputSchema(TypedDict):
    query:str

class OutputSchema(TypedDict):
    final_answer:str

def rag_search_node(state:MyState):
    query = state['query']
    rag_result = f"这是基于rag的搜索结果:{query}"
    return {
        "rag_result":rag_result
    }

def web_search_node(state:MyState):
    query = state['query']
    web_result = f"这是基于web的搜索结果:{query}"
    return {
        "web_result":web_result
    }

# 最终节点需要接受rag和web的搜索结果
def final_answer_node(state:PrivateState):
    rag_result = state['rag_result']
    web_result = state['web_result']
    final_answer = f"这是最终的答案: {rag_result} {web_result}"
    return {
        "final_answer":final_answer
    }

builder = StateGraph(state_schema=MyState,
                     input_schema=InputSchema,
                     output_schema=OutputSchema)
builder.add_node(rag_search_node)
builder.add_node(web_search_node)
builder.add_node(final_answer_node)
builder.add_edge(START,"rag_search_node")
builder.add_edge("rag_search_node", "web_search_node")
builder.add_edge("web_search_node", "final_answer_node")
graph = builder.compile()
res = graph.invoke({"query":"如何使用langgraph"})
print(res)





{'final_answer': '这是最终的答案: 这是基于rag的搜索结果:如何使用langgraph 这是基于web的搜索结果:如何使用langgraph'}
